# Evaluating CFU model on entropy dataset

## Load data

Configure root.

In [ ]:
import sys, subprocess
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
%matplotlib inline
from pathlib import Path

# Configure root
COLAB = Path("/content").exists()
repo_url = "https://github.com/eddykang06/phenotype-prediction.git"
repo_dir = Path("phenotype-prediction")
if COLAB:
    root = Path("/content/phenotype-prediction")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", repo_url])
else:
    root = Path.cwd().parent
sys.path.insert(0, str(root))

Configure data path.

In [ ]:
if COLAB:
  from google.colab import drive
  drive.mount("/content/drive")
  data_dir = Path("/content/drive/MyDrive/phenotype-prediction-data")
  fcnts_path = str(data_dir / "fcnts_timezero")
  cfu_path = str(data_dir /  "cfus")
  annot_path = str(data_dir / "Annotation_TIGR4.tsv") 
  entropy_fcnts_path = str(data_dir / "entropy_data" / "fcnts")
  entropy_od_path = str(data_dir / "entropy_data" / "od600" / "growth_curves.csv")

else:
  data_dir = Path("C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab")
  fcnts_path = str(data_dir / "fcnts_timezero")
  cfu_path = str(data_dir / "cfus")
  annot_path = str(data_dir / "Annotation_TIGR4.tsv")
  entropy_fcnts_path = str(data_dir / "entropy_data" / "fcnts")
  entropy_od_path = str(data_dir / "entropy_data" / "od600" / "growth_curves.csv")


Load TPM data from entropy dataset.

In [ ]:
from src.tpm_data import (
    fcnts_to_tpms, 
    read_fcnts_as_df, 
    bind_tpm_data,
    sample_name_strip
)

data_df = bind_tpm_data(fcnts_to_tpms(read_fcnts_as_df(entropy_fcnts_path, entropy = True)))
data_df.index = [sample_name_strip(x) for x in data_df.index]

Load OD600 data and convert to CFUs.

In [ ]:
import pandas as pd

df = pd.read_csv(entropy_od_path, header=[0, 1], index_col = 0)

df.index.name = "time_min"
df.columns.names = ["drug", "replicate"]

long_df = (
    df.stack(["drug", "replicate"], future_stack = True)
      .rename("OD600")
      .dropna()
      .reset_index()
)

long_df["drug_id"] = (
    long_df["drug"]
    + long_df["time_min"].astype(str)
    + "min-"
    + long_df["replicate"]
)

long_df = long_df.set_index("drug_id")[["OD600"]]

# Convert OD to CFU
long_df["CFU"] = long_df["OD600"] * 3.3 * 50 * 10**6
long_df = long_df.drop(columns = ["OD600"])

# Bind to TPM data
data_df = pd.merge(data_df, long_df, left_index = True, right_index = True, how = "inner")

In [ ]:
data_df

Extract metdata.

In [ ]:
from src.metadata import condition_to_drug_id, condition_to_timepoint

index = data_df.index
meta = pd.DataFrame(
    {
    "drug_id": [condition_to_drug_id(x) for x in index],
    "timepoint": [condition_to_timepoint(x) for x in index],
    "drug1_dose": [1]*len(index)
    }, 
    index = index
)

## Model predictions

In [ ]:
df = pd.read_csv("growth_curves.csv", header=[0, 1])